# Bronze Layer — OpenAlex Works Ingestion

This notebook ingests raw OpenAlex Works data from a Unity Catalog volume into a Delta table 
in `openalex_lakehouse.bronze`.

## What this notebook does:
1. Reads raw OpenAlex data as JSON lines from Unity Catalog volume
2. Stores each record as a raw JSON string without parsing
3. Adds ingestion metadata columns (ingestion_date, source_file)
4. Writes data to a Delta table in Unity Catalog for downstream processing

### Configuration

In [0]:
# Configuration
from pyspark.sql.functions import current_timestamp, col

CATALOG = "openalex_lakehouse"
SCHEMA = "bronze"
TABLE = "works_raw"

TARGET_TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE}"

INPUT_PATH = "/Volumes/openalex_lakehouse/bronze/raw_files/open_alex_split_500_records.json"

print(f"Raw Input path: {INPUT_PATH}")


### Read Raw Data

In [0]:
# Reading raw JSON files from the Volume

raw_df = (spark.read
          .text(INPUT_PATH)
          .withColumnRenamed("value", "raw_json"))

raw_df.show()
print(raw_df.count())

### Add metadata

In [0]:
bronze_df = (raw_df.withColumn("ingestion_date", current_timestamp())
                  .withColumn("source_file", col("_metadata.file_path"))
)

bronze_df.show()

In [0]:
dfRecordCount = bronze_df.count()
print(f"Total number of records: {dfRecordCount}")

In [0]:
bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(TARGET_TABLE_NAME)

In [0]:
# Check nulls
nullCount = bronze_df.filter("raw_json IS NULL").count()
print(f"Records with null value: {nullCount}")

dbRecordCount = spark.sql("SELECT COUNT(*) FROM openalex_lakehouse.bronze.works_raw").collect()[0][0]

if dfRecordCount == dbRecordCount:
   print(f"Data ingestion is successful. Dataframe record count: {dfRecordCount}, DB Record Count: {dbRecordCount}")
else:
    print(f"Count don't match. Dataframe: {dfRecordCount}, DB Record Count: {dbRecordCount}")